In [ ]:
# pip 어떤 버전이 어떤 가상환경에 설치되어있는지 확인
#!pip --version

pip 26.2.1 from D:\hykim\hanwha_0902\.venv\Lib\site-packages\pip (python 3.12)



In [ ]:
# 패키지 : 설치한 사용라이브러리 묶음
# !pip install python-dotenv

In [ ]:
# LangSmith 추적을 설정합니다.
#!pip install -U langchain langchain-openai

In [ ]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv
import os

# API KEY 정보로드
# .env 파일 로드 (override=False가 기본값이므로 시스템 환경변수가 우선순위를 가집니다)
load_dotenv()

# 정상 로드 확인 테스트
# 설정된 키값을 출력: 키값이 노출되지 않도록 앞부분(8글자만) 출력해본다.
print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH_API_KEY:", os.getenv("LANGSMITH_API_KEY")[:8]+"...")
print("LANGSMITH_PROJECT:", os.getenv("LANGSMITH_PROJECT"))
print("LANGSMITH_ENDPOINT:", os.getenv("LANGSMITH_ENDPOINT"))

OPENAI_API_KEY: sk-proj-...
LANGSMITH_API_KEY: lsv2_pt_...
LANGSMITH_PROJECT: hanwha_0902
LANGSMITH_ENDPOINT: https://api.smith.langchain.com


## 프롬프트 템플릿의 활용

`PromptTemplate`

- 사용자의 입력 변수를 사용하여 완전한 프롬프트 문자열을 만드는 데 사용되는 템플릿입니다
- 사용법
  - `template`: 템플릿 문자열입니다. 이 문자열 내에서 중괄호 `{}`는 변수를 나타냅니다.
  - `input_variables`: 중괄호 안에 들어갈 변수의 이름을 리스트로 정의합니다.

`input_variables`

- input_variables는 PromptTemplate에서 사용되는 변수의 이름을 정의하는 리스트입니다.

In [ ]:
#랭스미스 추척하기.
#logging.langsmith("hanwha_0902", set_enable=True)

#랭스미스 추척하지 않기.
#logging.langsmith("hanwha_0902", set_enable=False)

In [73]:
import logging
from langchain_teddynote import logging
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from langchain_teddynote.messages import stream_response
from langchain_openai import ChatOpenAI

In [74]:
logging.langsmith("hanwha_0902")

LangSmith 추적을 시작합니다.
[프로젝트명]
hanwha_0902


In [80]:
# LLM 객체 정의
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
)

### 방법 1. from_template() 메소드를 사용하여 PromptTemplate 객체 생성

- 치환될 변수를 `{ 변수 }` 로 묶어서 템플릿을 정의합니다.

In [81]:
from langchain_core.prompts import PromptTemplate

#templete 정의.{python_module}는 변수로, 이후에 값이 들어갈 자리를 의미
template = "파이썬에서 {python_module} 어떻게 사용되나요?"

# from_template 메소드를 이용하여 PromptTemplate 객체 생성
prompt = PromptTemplate.from_template(template)
prompt


PromptTemplate(input_variables=['python_module'], input_types={}, partial_variables={}, template='파이썬에서 {python_module} 어떻게 사용되나요?')

`python_modue` 변수에 값을 넣어서 문장을 생성할 수 있습니다.

In [82]:
# prompt 생성. format 메소드를 이용하여 변수에 값을 넣어줌
prompt = prompt.format(python_module="FastAPI")
prompt

'파이썬에서 FastAPI 어떻게 사용되나요?'

In [83]:
# template 정의
template = "파이썬에서 {python_module} 어떻게 사용되나요?"

# from_template 메소드를 이용하여 PromptTemplate 객체 생성
prompt = PromptTemplate.from_template(template)

# chain 생성
chain = prompt | llm


In [84]:
# 변수에 입력된 값이 자동으로 치환되어 실행됨
chain.invoke("FastAPI").content

'FastAPI는 Python으로 작성된 현대적인 웹 프레임워크로, 빠르고 효율적인 API를 구축하는 데 사용됩니다. FastAPI는 비동기 프로그래밍을 지원하며, 자동으로 OpenAPI 문서를 생성하는 기능이 있습니다. 아래는 FastAPI를 사용하는 기본적인 방법에 대한 단계별 설명입니다.\n\n### 1. FastAPI 설치\n\nFastAPI와 ASGI 서버인 `uvicorn`을 설치합니다. 터미널에서 다음 명령어를 실행하세요.\n\n```bash\npip install fastapi uvicorn\n```\n\n### 2. 기본 FastAPI 애플리케이션 생성\n\n아래는 FastAPI 애플리케이션의 기본 구조입니다.\n\n```python\n# main.py\nfrom fastapi import FastAPI\n\napp = FastAPI()\n\n@app.get("/")\nasync def read_root():\n    return {"Hello": "World"}\n\n@app.get("/items/{item_id}")\nasync def read_item(item_id: int, q: str = None):\n    return {"item_id": item_id, "q": q}\n```\n\n### 3. 애플리케이션 실행\n\n터미널에서 다음 명령어를 실행하여 FastAPI 애플리케이션을 실행합니다.\n\n```bash\nuvicorn main:app --reload\n```\n\n- `main`은 Python 파일의 이름 (확장자 제외)\n- `app`은 FastAPI 인스턴스의 이름\n- `--reload` 플래그는 코드 변경 시 자동으로 서버를 재시작합니다.\n\n### 4. API 테스트\n\n브라우저에서 `http://127.0.0.1:8000`에 접속하면 `{"Hello": "World"}`라는 JSON 응답을 받을 수 있습니다. 또한, `http://127.0.0.1:8000/items/1?q=test`에 접속하면 다음과 같은 

In [ ]:
# # prompt 생성. format 메소드를 이용하여 변수에 값을 넣어줌
# prompt = prompt_template.format(python_module="Streamlit")
# prompt

'파이썬에서 Streamlit 어떻게 사용되나요?'